# 3D Runtime Benchmark — GBM vs Baselines

This notebook measures wall-clock runtime for GBM-3D and several 3D baselines on synthetic data.

It is useful for:
- Reproducing runtime claims from the paper
- Understanding the performance trade-offs between methods
- Quick sanity checks when modifying code

In [ ]:
import numpy as np
import pandas as pd
import time
from contextlib import contextmanager

@contextmanager
def timer(name):
    start = time.perf_counter()
    yield
    end = time.perf_counter()
    print(f"{name:25s}: {end - start:.3f} s")
    return end - start

In [ ]:
# Small synthetic 3D volume for benchmarking
lw = 2
W, H_img, D = 100 * lw, 100, 10   # modest size for reasonable runtimes
x = np.linspace(-20, 20, W)
y = np.linspace(0.4, 12, D)
z = np.linspace(-70, 70, H_img)

X, Y, Z = np.meshgrid(x, y, z, indexing="ij")
C = 5e-6 + 3e-6 * np.exp(-((X)**2 + (Z)**2 + (Y-5)**2) / 65)

df = pd.DataFrame({
    "X": X.ravel(),
    "Y": Y.ravel(),
    "Z": Z.ravel(),
    "Carbon": C.ravel()
})

inside = np.ones(W * H_img * D, dtype=bool)
global_mean = float(C.mean())

print(f"Benchmark volume: {W}×{H_img}×{D}  (LW={lw})")
print(f"Global mean: {global_mean:.3e}")

In [ ]:
k = 6
results = []

print("\n=== 3D Runtime Benchmark (k={}) ===\n".format(k))

In [ ]:
# GBM 3D
try:
    from gbm.core import find_optimal_k_points_gbm_3D
    with timer("GBM-XZ (D-1)") as t:
        _ = find_optimal_k_points_gbm_3D(
            df, inside, k=k, in_CO2_avg=global_mean,
            cross_section="XZ", barn_LW_ratio=lw,
            epochs=6, sampling_budget=500
        )
    results.append(("GBM-XZ", t))
except Exception as e:
    print(f"GBM 3D failed: {e}")

In [ ]:
# Greedy 3D
try:
    from gbm.baselines.greedy import find_optimal_k_points_greedy_3D
    with timer("Greedy 3D") as t:
        _ = find_optimal_k_points_greedy_3D(
            df, inside, k=k, in_CO2_avg=global_mean, barn_LW_ratio=lw
        )
    results.append(("Greedy 3D", t))
except Exception as e:
    print(f"Greedy 3D failed: {e}")

In [ ]:
# Uniform Grid 3D
try:
    from gbm.baselines.uniform import find_optimal_k_points_uniform_grid_search_3D
    with timer("Uniform Grid 3D") as t:
        _ = find_optimal_k_points_uniform_grid_search_3D(
            df, inside, k=k, in_CO2_avg=global_mean, barn_LW_ratio=lw,
            max_subset_evals=1000
        )
    results.append(("Uniform 3D", t))
except Exception as e:
    print(f"Uniform 3D failed: {e}")

In [ ]:
# PSO 3D (limited epochs for benchmark)
try:
    from gbm.baselines.PSO import find_optimal_k_points_pso_3D
    with timer("PSO 3D") as t:
        _ = find_optimal_k_points_pso_3D(
            df, inside, k=k, in_CO2_avg=global_mean, barn_LW_ratio=lw, epochs=20
        )
    results.append(("PSO 3D", t))
except Exception as e:
    print(f"PSO 3D failed: {e}")

In [ ]:
# Monte Carlo 3D
try:
    from gbm.baselines.monte_carlo import find_optimal_k_points_monte_carlo_3D
    with timer("Monte Carlo 3D") as t:
        _ = find_optimal_k_points_monte_carlo_3D(
            df, inside, k=k, in_CO2_avg=global_mean, barn_LW_ratio=lw, max_epochs=15
        )
    results.append(("Monte Carlo 3D", t))
except Exception as e:
    print(f"Monte Carlo 3D failed: {e}")

In [ ]:
# Random 3D
try:
    from gbm.baselines.random import find_optimal_k_points_random_search_3D
    with timer("Random 3D") as t:
        _ = find_optimal_k_points_random_search_3D(
            df, inside, k=k, in_CO2_avg=global_mean, barn_LW_ratio=lw, epochs=300
        )
    results.append(("Random 3D", t))
except Exception as e:
    print(f"Random 3D failed: {e}")

In [ ]:
print("\n=== Runtime Summary (seconds) ===")
for name, t in sorted(results, key=lambda x: x[1]):
    print(f"{name:22s}: {t:.3f} s")

**Notes**
- All runs use synthetic data only.
- Epochs / sampling budgets are intentionally modest for a notebook.
- Real paper benchmarks used larger volumes and full configurations.
- This notebook is mainly for relative comparison and regression testing.